In [ ]:
# ================================================================
# ontrastive Likelihood Detection
# ================================================================
# Replaces raw perplexity with:
#   S(x) = log P_small(x) - log P_large(x)
#
# Intuition (distribution-shift theory):
#   LLM text is optimised under a large model → anomalously smooth
#   to large model but appears rough to small model → measurable gap.
#   Human text does NOT exhibit this systematic gap.
# ================================================================

# ── Cell 1: Install & Imports ──────────────────────────────────
!pip install -q transformers accelerate

import os, json, pickle
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from tqdm import tqdm
import warnings; warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

RESULTS_DIR = "./results/contrastive"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Cell 2: Model Scale Groups ─────────────────────────────────
# Three-scale ladder for multi-scale contrast.
# All are decoder-only causal LMs trained on similar corpora (OpenWebText / Pile).
SCALE_MODELS = {
    "small":  "gpt2",            # 117M
    "medium": "gpt2-medium",     # 345M
    "large":  "gpt2-xl",         # 1.5B  — use 4-bit on Colab T4/A100
}

# ── Cell 3: Load Data ──────────────────────────────────────────
def encode_labels(series):
    return (series == "llm").astype(int)

hc3_test  = pd.read_csv("hc3_test.csv")
eli5_test = pd.read_csv("eli5_test.csv")

for df in [hc3_test, eli5_test]:
    df["label_encoded"] = encode_labels(df["label"])

print(f"HC3  test : {len(hc3_test):,}")
print(f"ELI5 test : {len(eli5_test):,}")

# ── Cell 4: Token-Level Log-Prob Engine ───────────────────────

class TokenLogProbCalculator:
    """
    Computes per-token log-probabilities under a causal LM.
    Returns both document-level (mean) and per-token arrays,
    enabling token-level variance analysis.
    """

    def __init__(self, model_name: str, use_fp16: bool = True):
        self.model_name = model_name
        print(f"  Loading {model_name} ...")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        dtype = torch.float16 if (use_fp16 and torch.cuda.is_available()) else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            device_map="auto",
        )
        self.model.eval()
        print(f"  ✅ {model_name} ready")

    def get_token_log_probs(self, text: str, max_length: int = 512):
        """
        Returns:
            mean_log_prob   : float  — average log P(token | context)
            token_log_probs : ndarray — per-token log probabilities
            num_tokens      : int
        """
        enc = self.tokenizer(
            text,
            max_length=max_length,
            truncation=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].to(self.model.device)

        if input_ids.shape[1] < 2:
            return np.nan, np.array([]), 0

        with torch.no_grad():
            outputs = self.model(input_ids)
            logits  = outputs.logits  # (1, T, V)

        # Shift: predict token t+1 from context up to t
        shift_logits = logits[:, :-1, :]           # (1, T-1, V)
        shift_labels = input_ids[:, 1:]            # (1, T-1)

        log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
        token_lp  = log_probs[0, torch.arange(shift_labels.shape[1]), shift_labels[0]]
        token_lp_np = token_lp.cpu().float().numpy()

        return float(token_lp_np.mean()), token_lp_np, len(token_lp_np)

    def score_corpus(self, texts, batch_desc: str = ""):
        """Score a list of texts. Returns (mean_log_probs, token_lp_lists)."""
        mean_lps, token_lps_all = [], []
        for text in tqdm(texts, desc=f"  logP [{batch_desc}]", leave=False):
            m, t, _ = self.get_token_log_probs(str(text))
            mean_lps.append(m)
            token_lps_all.append(t)
        return np.array(mean_lps), token_lps_all

    def cleanup(self):
        del self.model
        torch.cuda.empty_cache()


# ── Cell 5: Compute Log-Probs for All Scales ──────────────────

log_probs_store = {}   # {scale: {dataset: (mean_lps, token_lps_all)}}

for scale, model_name in SCALE_MODELS.items():
    print(f"\n{'='*60}")
    print(f"Scale: {scale.upper()}  ({model_name})")
    print(f"{'='*60}")

    calc = TokenLogProbCalculator(model_name)
    log_probs_store[scale] = {}

    for ds_name, df in [("hc3", hc3_test), ("eli5", eli5_test)]:
        mean_lps, token_lps = calc.score_corpus(df["text"].tolist(), f"{scale}/{ds_name}")
        log_probs_store[scale][ds_name] = {
            "mean_lps":   mean_lps,
            "token_lps":  token_lps,
            "labels":     df["label_encoded"].values,
        }
        print(f"  {ds_name}: mean_log_prob = {np.nanmean(mean_lps):.4f}")

    calc.cleanup()

# ── Cell 6: Contrastive Score Derivations ─────────────────────

def contrastive_score(small_lps, large_lps):
    """
    S(x) = log P_small(x) - log P_large(x)
    Positive → small model finds text *relatively* more probable than large.
    Human text: gap ≈ 0 or random.
    LLM text:   large model assigns much higher prob → S(x) is more negative.
    We negate so higher score = more likely AI.
    """
    score = -(small_lps - large_lps)   # negate so higher = more AI-like
    return score


def multi_scale_score(small_lps, medium_lps, large_lps):
    """
    Combine three pairwise contrasts:
      S_12 = small vs medium
      S_23 = medium vs large
      S_13 = small vs large
    Final = weighted average (equal weights here).
    """
    s12 = -(small_lps - medium_lps)
    s23 = -(medium_lps - large_lps)
    s13 = -(small_lps - large_lps)
    return (s12 + s23 + s13) / 3.0


def token_contrast_variance(small_token_lps, large_token_lps):
    """
    Per-document: variance of (log P_large(t) - log P_small(t)) across tokens.
    LLM text tends to have more uniform token-level likelihoods under the
    large model (smooth generation) → lower variance. Human text is noisier.
    Returns: negated variance so higher = more AI-like.
    """
    scores = []
    for s_tok, l_tok in zip(small_token_lps, large_token_lps):
        min_len = min(len(s_tok), len(l_tok))
        if min_len < 2:
            scores.append(np.nan)
            continue
        diff    = l_tok[:min_len] - s_tok[:min_len]
        scores.append(float(np.var(diff)))
    # Negate so higher score = lower variance = more AI-like
    arr = np.array(scores)
    return -arr


def rankscale(arr):
    """Map raw scores → [0, 1] via rank normalisation."""
    valid = ~np.isnan(arr)
    out   = np.full_like(arr, fill_value=np.nan, dtype=float)
    ranked = np.argsort(np.argsort(arr[valid]))
    out[valid] = ranked / (ranked.max() + 1e-9)
    return out


# ── Cell 7: Build All Score Variants ──────────────────────────

score_variants = {}   # {variant_name: {dataset: {scores, labels}}}

for ds_name in ["hc3", "eli5"]:
    s = log_probs_store["small"][ds_name]
    m = log_probs_store["medium"][ds_name]
    l = log_probs_store["large"][ds_name]

    labels = s["labels"]

    # Variant 1: Base contrastive (small vs large)
    raw_base   = contrastive_score(s["mean_lps"], l["mean_lps"])
    base_scores = rankscale(raw_base)
    score_variants.setdefault("base_contrast", {})[ds_name] = {
        "scores": base_scores, "raw": raw_base, "labels": labels}

    # Variant 2: Multi-scale (all three)
    raw_ms   = multi_scale_score(s["mean_lps"], m["mean_lps"], l["mean_lps"])
    ms_scores = rankscale(raw_ms)
    score_variants.setdefault("multi_scale", {})[ds_name] = {
        "scores": ms_scores, "raw": raw_ms, "labels": labels}

    # Variant 3: Token contrast variance
    raw_var   = token_contrast_variance(s["token_lps"], l["token_lps"])
    var_scores = rankscale(raw_var)
    score_variants.setdefault("token_variance", {})[ds_name] = {
        "scores": var_scores, "raw": raw_var, "labels": labels}

    # Variant 4: Hybrid — mean of all three normalised scores
    hybrid = np.nanmean(np.stack([base_scores, ms_scores, var_scores]), axis=0)
    score_variants.setdefault("hybrid", {})[ds_name] = {
        "scores": hybrid, "raw": hybrid, "labels": labels}

    print(f"\n[{ds_name}] Score shapes ok ✅")

# ── Cell 8: Metrics ────────────────────────────────────────────
print("\n" + "="*70)
print("CONTRASTIVE DETECTOR PERFORMANCE")
print("="*70)

rows = []
for variant, ds_dict in score_variants.items():
    for ds_name, data in ds_dict.items():
        valid = ~np.isnan(data["scores"])
        y_t   = data["labels"][valid]
        y_s   = data["scores"][valid]

        if len(np.unique(y_t)) < 2:
            continue

        auc    = roc_auc_score(y_t, y_s)
        brier  = brier_score_loss(y_t, y_s)
        sep    = y_s[y_t==1].mean() - y_s[y_t==0].mean()

        rows.append({
            "Variant":         variant,
            "Dataset":         ds_name,
            "ROC-AUC":         auc,
            "Brier Score":     brier,
            "Mean Human Score":y_s[y_t==0].mean(),
            "Mean LLM Score":  y_s[y_t==1].mean(),
            "Score Separation":sep,
        })
        print(f"  {variant:20s} [{ds_name}]  AUC={auc:.4f}  Sep={sep:.4f}")

summary_df = pd.DataFrame(rows)
summary_df.to_csv(f"{RESULTS_DIR}/contrastive_summary.csv", index=False)
print(f"\n✅ Saved: contrastive_summary.csv")

# ── Cell 9: Distribution Plots ─────────────────────────────────
variants_list = list(score_variants.keys())
datasets_list = ["hc3", "eli5"]

fig, axes = plt.subplots(len(variants_list), 2, figsize=(14, 4*len(variants_list)))
fig.suptitle("Contrastive Likelihood: Detectability Score Distributions",
             fontsize=15, fontweight="bold")

for i, variant in enumerate(variants_list):
    for j, ds_name in enumerate(datasets_list):
        ax  = axes[i, j]
        data = score_variants[variant][ds_name]
        valid = ~np.isnan(data["scores"])
        y_t  = data["labels"][valid]
        y_s  = data["scores"][valid]

        ax.hist(y_s[y_t==0], bins=40, alpha=0.6, label="Human",
                color="#3498db", range=(0,1))
        ax.hist(y_s[y_t==1], bins=40, alpha=0.6, label="LLM",
                color="#e74c3c", range=(0,1))

        auc = roc_auc_score(y_t, y_s) if len(np.unique(y_t)) > 1 else float("nan")
        ax.set_title(f"{variant} | {ds_name.upper()}  AUC={auc:.3f}",
                     fontsize=11, fontweight="bold")
        ax.set_xlabel("Detectability Score (rank-normalised)")
        ax.set_ylabel("Frequency")
        ax.axvline(0.5, color="k", linestyle="--", alpha=0.4)
        ax.legend(); ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/contrastive_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Cell 10: Calibration Curves ────────────────────────────────
fig, axes = plt.subplots(len(variants_list), 2, figsize=(14, 4*len(variants_list)))
fig.suptitle("Contrastive Likelihood: Calibration", fontsize=15, fontweight="bold")

for i, variant in enumerate(variants_list):
    for j, ds_name in enumerate(datasets_list):
        ax   = axes[i, j]
        data = score_variants[variant][ds_name]
        valid = ~np.isnan(data["scores"])
        y_t   = data["labels"][valid]
        y_s   = data["scores"][valid]

        if len(np.unique(y_t)) > 1:
            fop, mpv = calibration_curve(y_t, y_s, n_bins=10, strategy="uniform")
            ax.plot(mpv, fop, "s-", linewidth=2, color="#2ecc71", label="Detector")

        ax.plot([0,1],[0,1], "k--", alpha=0.5, label="Perfect")
        ax.set_title(f"{variant} | {ds_name.upper()}", fontsize=11, fontweight="bold")
        ax.set_xlabel("Mean Predicted"); ax.set_ylabel("Fraction LLM")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/contrastive_calibration.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Cell 11: ROC Curves ────────────────────────────────────────
fig, axes = plt.subplots(len(variants_list), 2, figsize=(14, 4*len(variants_list)))
fig.suptitle("Contrastive Likelihood: ROC Curves", fontsize=15, fontweight="bold")

for i, variant in enumerate(variants_list):
    for j, ds_name in enumerate(datasets_list):
        ax   = axes[i, j]
        data = score_variants[variant][ds_name]
        valid = ~np.isnan(data["scores"])
        y_t   = data["labels"][valid]
        y_s   = data["scores"][valid]

        if len(np.unique(y_t)) > 1:
            fpr, tpr, _ = roc_curve(y_t, y_s)
            auc = roc_auc_score(y_t, y_s)
            ax.plot(fpr, tpr, linewidth=2.5, color="#9b59b6",
                    label=f"AUC={auc:.3f}")

        ax.plot([0,1],[0,1], "k--", alpha=0.4)
        ax.set_title(f"{variant} | {ds_name.upper()}", fontsize=11, fontweight="bold")
        ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
        ax.legend(fontsize=9, loc="lower right"); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/contrastive_roc.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Cell 12: Token-Level Contrast Visualisation ────────────────
# Show average per-token contrast profile for human vs LLM text.
print("\n" + "="*70)
print("TOKEN-LEVEL CONTRAST PROFILE ANALYSIS")
print("="*70)

for ds_name in ["hc3", "eli5"]:
    data = score_variants["base_contrast"][ds_name]
    s_store = log_probs_store["small"][ds_name]
    l_store = log_probs_store["large"][ds_name]
    labels  = data["labels"]

    # Truncate to first 100 tokens for alignment
    T = 100
    human_profiles, llm_profiles = [], []

    for idx in range(len(labels)):
        s_tok = s_store["token_lps"][idx]
        l_tok = l_store["token_lps"][idx]
        min_len = min(len(s_tok), len(l_tok), T)
        if min_len < 10:
            continue
        diff = l_tok[:min_len] - s_tok[:min_len]
        padded = np.full(T, np.nan)
        padded[:min_len] = diff
        if labels[idx] == 0:
            human_profiles.append(padded)
        else:
            llm_profiles.append(padded)

    if human_profiles and llm_profiles:
        h_mean = np.nanmean(human_profiles, axis=0)
        l_mean = np.nanmean(llm_profiles,   axis=0)

        plt.figure(figsize=(12, 4))
        plt.plot(h_mean, label="Human (mean)", color="#3498db", linewidth=2)
        plt.plot(l_mean, label="LLM (mean)",   color="#e74c3c", linewidth=2)
        plt.fill_between(range(T), h_mean, l_mean, alpha=0.15, color="grey")
        plt.xlabel("Token Position")
        plt.ylabel("log P_large − log P_small (per token)")
        plt.title(f"Token-Level Contrast Profile — {ds_name.upper()}\n"
                  f"Positive = large model finds token more probable than small model",
                  fontsize=12)
        plt.legend(); plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(f"{RESULTS_DIR}/token_contrast_{ds_name}.png", dpi=150)
        plt.show()

# ── Cell 13: Baseline Comparison (vs raw perplexity) ──────────
print("\n" + "="*70)
print("COMPARISON: CONTRASTIVE vs RAW PERPLEXITY (Stage 2C)")
print("="*70)

# Attempt to load Stage 2C results if available
try:
    ppl_df = pd.read_csv("detector_family3_perplexity_results.csv")
    # Align naming
    ppl_df["Variant"] = "ppl_" + ppl_df["Detector"].str.replace("PPL-", "")
    compare = pd.concat([
        summary_df[["Variant","Dataset","ROC-AUC"]].rename(
            columns={"Variant":"Method","ROC-AUC":"AUC"}),
        ppl_df[["Variant","Evaluation","ROC-AUC"]].rename(
            columns={"Variant":"Method","Evaluation":"Dataset","ROC-AUC":"AUC"})
    ])
    pivot = compare.pivot_table(index="Method", columns="Dataset", values="AUC")
    print(pivot.round(4))
except FileNotFoundError:
    print("  Stage 2C results not found — run Stage 2C first for full comparison.")

# Save full results
with open(f"{RESULTS_DIR}/contrastive_score_variants.pkl", "wb") as f:
    pickle.dump({v: {k: {"scores": d["scores"].tolist(),
                          "labels": d["labels"].tolist()}
                     for k, d in ds.items()}
                 for v, ds in score_variants.items()}, f)

print(f"\n✅ All results saved to {RESULTS_DIR}/")
print("🎯 Contrastive Likelihood Detection complete.")